# FFmpeg Worker Loop

This notebook implements a worker loop that:
1. Fetches encoding tasks from `/api/encode/largest`
2. Builds and executes the ffmpeg command
3. Reports results back to `/api/encoded`

In [ ]:
import requests
import subprocess
import time
from pathlib import Path
import os

## Configuration

Set the API base URL and worker settings

In [ ]:
# API Configuration
API_BASE_URL = "http://localhost:5000"  # Update this to match your Flask server
GET_TASK_ENDPOINT = f"{API_BASE_URL}/api/encode/largest"
POST_RESULT_ENDPOINT = f"{API_BASE_URL}/api/encoded"

# Worker Configuration
POLL_INTERVAL = 60  # Seconds to wait between polls when no task is available

## Helper Functions

Functions to fetch tasks, run ffmpeg, and report results

In [ ]:
def fetch_encoding_task():
    """
    Fetch the largest encoding task from the API
    Returns: tuple (success, data/error_message)
    """
    try:
        print(f"[FETCH] Requesting task from {GET_TASK_ENDPOINT}")
        response = requests.get(GET_TASK_ENDPOINT, timeout=10)
        response.raise_for_status()
        
        result = response.json()
        
        if result.get('success') and result.get('data'):
            print(f"[FETCH] Task received - GUID: {result['data'].get('guid')}")
            print(f"[FETCH] File: {result['data'].get('input_file_name')}")
            print(f"[FETCH] Size: {result['data'].get('before_file_size')} bytes")
            return True, result['data']
        else:
            print("[FETCH] No tasks available")
            return False, result.get('message', 'No tasks available')
            
    except requests.exceptions.RequestException as e:
        print(f"[ERROR] Failed to fetch task: {e}")
        return False, str(e)

In [ ]:
def run_ffmpeg(task_data):
    """
    Execute ffmpeg command from task data
    Returns: tuple (success, output_file_size/error_message)
    """
    try:
        # Extract paths and command
        directory = task_data.get('directory_path')
        input_file = task_data.get('input_file_name')
        output_file = task_data.get('output_file_name')
        ffmpeg_string = task_data.get('ffmpeg_string')
        
        if not all([directory, input_file, output_file, ffmpeg_string]):
            return False, "Missing required fields in task data"
        
        # Build full paths
        input_path = Path(directory) / input_file
        output_path = Path(directory) / output_file
        
        # Verify input file exists
        if not input_path.exists():
            return False, f"Input file not found: {input_path}"
        
        # Build the complete ffmpeg command
        # The ffmpeg_string from DB should contain the full command
        # Replace placeholders if needed
        ffmpeg_cmd = ffmpeg_string.replace('{input}', str(input_path))
        ffmpeg_cmd = ffmpeg_cmd.replace('{output}', str(output_path))
        
        print(f"[FFMPEG] Input: {input_path}")
        print(f"[FFMPEG] Output: {output_path}")
        print(f"[FFMPEG] Command: {ffmpeg_cmd}")
        print(f"[FFMPEG] Starting encoding...")
        
        # Run ffmpeg command
        result = subprocess.run(
            ffmpeg_cmd,
            shell=True,
            capture_output=True,
            text=True
        )
        
        if result.returncode != 0:
            print(f"[ERROR] FFmpeg failed with return code {result.returncode}")
            print(f"[ERROR] stderr: {result.stderr[:500]}")  # Print first 500 chars of error
            return False, f"FFmpeg failed: {result.stderr[:200]}"
        
        # Verify output file was created
        if not output_path.exists():
            return False, "Output file was not created"
        
        # Get output file size
        output_size = output_path.stat().st_size
        print(f"[FFMPEG] Encoding complete!")
        print(f"[FFMPEG] Output size: {output_size} bytes")
        
        return True, output_size
        
    except Exception as e:
        print(f"[ERROR] Exception during ffmpeg execution: {e}")
        return False, str(e)

In [ ]:
def report_results(guid, after_file_size):
    """
    Report encoding results back to the API
    Returns: tuple (success, response_data/error_message)
    """
    try:
        payload = {
            'guid': guid,
            'after_file_size': after_file_size
        }
        
        print(f"[REPORT] Posting results to {POST_RESULT_ENDPOINT}")
        print(f"[REPORT] GUID: {guid}, After size: {after_file_size} bytes")
        
        response = requests.post(
            POST_RESULT_ENDPOINT,
            json=payload,
            timeout=10
        )
        response.raise_for_status()
        
        result = response.json()
        
        if result.get('success'):
            print(f"[REPORT] Results reported successfully!")
            return True, result
        else:
            print(f"[ERROR] API returned error: {result.get('error')}")
            return False, result.get('error', 'Unknown error')
            
    except requests.exceptions.RequestException as e:
        print(f"[ERROR] Failed to report results: {e}")
        return False, str(e)

## Worker Loop

Main processing loop that continuously fetches tasks, processes them, and reports results

In [ ]:
def worker_loop():
    """
    Main worker loop that processes encoding tasks
    """
    iteration = 0

    print("="*80)
    print("FFmpeg Worker Started")
    print("="*80)
    print(f"API Base URL: {API_BASE_URL}")
    print(f"Poll Interval: {POLL_INTERVAL}s")
    print(f"Max Iterations: Infinite")
    print("="*80)

    try:
        while True:
            iteration += 1
            print(f"\n{'='*80}")
            print(f"[WORKER] Iteration {iteration}")
            print(f"{'='*80}")
            
            # Step 1: Fetch a task
            success, task_data = fetch_encoding_task()
            
            if not success:
                print(f"[WORKER] No task available, waiting {POLL_INTERVAL}s...")
                time.sleep(POLL_INTERVAL)
                continue
            
            # Extract GUID for tracking
            guid = task_data.get('guid')
            
            # Step 2: Run ffmpeg
            success, result = run_ffmpeg(task_data)
            
            if not success:
                print(f"[WORKER] FFmpeg failed: {result}")
                print(f"[WORKER] Skipping result reporting for GUID: {guid}")
                # Wait before next iteration to avoid rapid failures
                time.sleep(POLL_INTERVAL)
                continue
            
            output_size = result
            
            # Step 3: Report results
            success, response = report_results(guid, output_size)
            
            if not success:
                print(f"[WORKER] Failed to report results: {response}")
            else:
                # Calculate space saved
                before_size = task_data.get('before_file_size', 0)
                space_saved = before_size - output_size
                savings_percent = (space_saved / before_size * 100) if before_size > 0 else 0
                
                print(f"\n{'='*80}")
                print(f"[SUCCESS] Task completed!")
                print(f"[SUCCESS] Before: {before_size:,} bytes")
                print(f"[SUCCESS] After: {output_size:,} bytes")
                print(f"[SUCCESS] Saved: {space_saved:,} bytes ({savings_percent:.1f}%)")
                print(f"{'='*80}")
            
            # Small delay before next iteration
            time.sleep(1)
    
    except KeyboardInterrupt:
        print("\n\n[WORKER] Interrupted by user, shutting down...")
    except Exception as e:
        print(f"\n[ERROR] Unexpected error in worker loop: {e}")
        raise
    finally:
        print("\n[WORKER] Worker stopped")
        print(f"[WORKER] Total iterations: {iteration}")

## Start the Worker

Run the worker loop. Set `max_iterations` to limit the number of tasks, or leave as `None` for continuous processing.

In [ ]:
# Start the worker
# Continuous processing: press the stop button (■) to interrupt the worker

worker_loop()

## Testing Individual Functions

Test individual components before running the full worker loop

In [ ]:
# Test fetching a task
success, data = fetch_encoding_task()
if success:
    print("\nTask Data:")
    for key, value in data.items():
        print(f"  {key}: {value}")